In [2]:
# Core scverse libraries
import scanpy as sc
import anndata as ad
import pandas as pd
import matplotlib.pyplot as plt
import scanpy.external as sce
import json

In [3]:
sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()
sc.settings.set_figure_params(dpi=80, facecolor="white")

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.10.3 anndata==0.8.0 umap==0.5.6 numpy==1.26.4 scipy==1.12.0 pandas==2.1.4 scikit-learn==1.5.2 statsmodels==0.14.4 igraph==0.11.6 pynndescent==0.5.12


In [4]:
adata = sc.read_h5ad("/Users/takahiro/Desktop/project/Reha/snRNAseq_scanpy/adata/Epi_Ctrl_6W.h5ad")

In [5]:
adata.obs["type"] = adata.obs["type"].map({"Ctrl":"Ctrl","Reha6W":"Ex","SED6W":"SED"})

In [6]:
adata

AnnData object with n_obs × n_vars = 416 × 24225
    obs: 'sample', 'type', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'doublet_score', 'predicted_doublet', 'leiden', 'cell_type'
    var: 'n_cells', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts'
    uns: 'cell_type_colors', 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'sample_colors', 'scrublet', 'umap'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    obsp: 'connectivities', 'distances'

In [87]:
sc.pl.highest_expr_genes(adata, n_top=20)

normalizing counts per cell
    finished (0:00:00)


In [88]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='sample', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='sample', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='sample', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [89]:
sc.pl.violin(
    adata,
    ["pct_counts_mt"],
    jitter=0,
    multi_panel=True,
)

In [90]:
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts")

In [91]:
sc.pp.normalize_total(adata, target_sum=1e4)

normalizing counts per cell
    finished (0:00:00)


In [92]:
sc.pp.log1p(adata)

In [93]:
adata.raw = adata.copy()

In [94]:
sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])

regressing out ['total_counts', 'pct_counts_mt']
    sparse input is densified and may lead to high memory use
    finished (0:00:12)


In [95]:
sc.pp.scale(adata, max_value=10)

In [96]:
sc.tl.pca(adata, svd_solver="arpack")

computing PCA
    with n_comps=50
    finished (0:00:48)


In [97]:
sc.pl.pca_variance_ratio(adata, log=True)

In [98]:
adata.write_h5ad("./adata/Epi_Ctrl_6W_beforecuration_241025.h5ad")

In [7]:
adata = sc.read_h5ad("./adata/Epi_Ctrl_6W_beforecuration_241025.h5ad")

# Over clustering

In [8]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15)

computing neighbors
    using 'X_pca' with n_pcs = 15
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:01)


In [9]:
sc.tl.umap(adata, spread=0.5, min_dist=0.5)

computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:01)


In [10]:
sc.pl.umap(adata, color=["type"], vmax=5)

In [11]:
sc.tl.leiden(adata, resolution=1)

running Leiden clustering
    finished: found 9 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_8397/1886685787.py:1: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(adata, resolution=1)


In [12]:
sc.pl.umap(adata, color=["leiden"])

In [13]:
# for figure
# Create a figure with subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Plotting the individual graphs
sc.pl.violin(adata, ['n_genes_by_counts'], groupby='leiden', ax=axes[0], show=False)
sc.pl.violin(adata, ['total_counts'], groupby='leiden', ax=axes[1], show=False)
sc.pl.violin(adata, ['pct_counts_mt'], groupby='leiden', ax=axes[2], show=False)

# Adjusting layout
plt.tight_layout()

# Display the plot
plt.show()

In [14]:
sc.pl.dotplot(adata, [
  "Ttn", "Mybpc3", #CMs
  "Dcn", "Col1a1", "Postn", # FBs
  "Frmd3", "Dlc1", "Myh11", # SMCs
  "Pecam1", "Cdh5", "Vwf","Npr3", # Enforthelial, cardialcells
  "Ptprc", "Mrc1", "Cd163", # Macrophages
  "Cd3e","Skap1", "Cd79a", "Cd79b","Il7r", "Kit", # Tcell Bcell
  "Lmnb1","Slpi","Retnlg","S100a9", #Granulocytes
  "Nrxn1", "Nrxn3", "Upk3b","Msln","Gpc3", 
  "Plin1", "Xkr4", "Acta2","Ms4a1","Ncr1","Vtn","Colec11","Steap4","Kcnj8","Mmrn1","Flt4","Vegfc"
], groupby="leiden", vmax=4)

In [15]:
adata = adata[adata.obs["leiden"]!="7"]

# Down stream

In [16]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=15)
sc.tl.umap(adata, spread=1, min_dist=1)

computing neighbors
    using 'X_pca' with n_pcs = 15
    finished: added to `.uns['neighbors']`
    `.obsp['distances']`, distances for each pair of neighbors
    `.obsp['connectivities']`, weighted adjacency matrix (0:00:00)
computing UMAP
    finished: added
    'X_umap', UMAP coordinates (adata.obsm)
    'umap', UMAP parameters (adata.uns) (0:00:00)


In [17]:
sc.tl.leiden(adata, resolution=0.2)

running Leiden clustering
    finished: found 3 clusters and added
    'leiden', the cluster labels (adata.obs, categorical) (0:00:00)


In [18]:
sc.pl.umap(adata, color=["type"])

In [21]:
adata.obs["leiden"] = adata.obs["leiden"].astype(str)
adata.obs.loc[adata.obs["leiden"]=="2","leiden"] = "0"

In [22]:
adata.obs["leiden"] = adata.obs["leiden"].astype("category")

In [23]:
adata.obs["leiden"]

GTCGAATCACCATATG-1    0
TCGGGCACAGCGTGCT-1    0
CGAGGCTGTAACTTCG-1    0
CTGATCCAGCTTCGTA-1    0
AGACCATCATCATCCC-1    0
                     ..
CTGCATCCATGTGGCC-1    1
AAGAACAAGTCTGCGC-1    1
TTCCGGTCATGAATAG-1    1
TCTTGCGGTCTTGAGT-1    1
GTCCCATCATGGGCAA-1    1
Name: leiden, Length: 389, dtype: category
Categories (2, object): ['0', '1']

In [122]:
pal=["#D55E00","#ADD8E6"]

In [123]:
import matplotlib.pyplot as plt
import pandas as pd

# 'type'と'leiden'ごとの集計を行い、割合を計算
type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)
type_leiden_percentage = type_leiden_counts.div(type_leiden_counts.sum(axis=1), axis=0) * 100

# 積み上げ棒グラフの作成
type_leiden_percentage.plot(kind='bar', stacked=True, figsize=(6, 5),color=pal)

# グラフのタイトルとラベルを設定
plt.title('Leiden Proportions per sample')
plt.xlabel('Type')
plt.ylabel('Percentage')

# 凡例を調整
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)
# グラフを表示
plt.tight_layout()
plt.show()


/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_23193/386264099.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  type_leiden_counts = adata.obs.groupby(['type', 'leiden']).size().unstack(fill_value=0)


In [125]:
sc.pl.umap(adata, color=["leiden"],use_raw=False,palette=pal)

In [25]:
sc.pl.violin(adata, ["Sema3c","Sema3d","Vegfc"], groupby="leiden", use_raw=True)

In [26]:
import numpy as np

print("adata.X range:", np.min(adata.X), "→", np.max(adata.X))
if adata.raw is not None:
    print("adata.raw.X range:", np.min(adata.raw.X), "→", np.max(adata.raw.X))


adata.X range: -9.869463318910347 → 10.0
adata.raw.X range: 0.0 → 7.7052193121025185


In [27]:
sc.pl.violin(adata, ["Sema3c","Sema3d","Vegfc"], groupby="leiden", use_raw=True)

In [28]:
adata.obs["type"].value_counts()

type
Ex      295
SED      83
Ctrl     11
Name: count, dtype: int64

In [32]:
adata.raw.X

<389x24225 sparse matrix of type '<class 'numpy.float64'>'
	with 540852 stored elements in Compressed Sparse Row format>

In [109]:
import numpy as np
from scipy.stats import kruskal, mannwhitneyu
import itertools

def test_gene_expression(adata, gene, groups=["0","1"]):
    """
    gene 発現の群間差をKruskal-WallisとMann-Whitneyで検定
    """
    # 各群ごとのデータを抽出
    grouped = [
        adata.raw[:, gene].X[adata.obs["leiden"] == g].toarray().flatten()
        for g in groups
    ]
    
    # 群間差 (全体比較)
    stat, pval = kruskal(*grouped)
    print(f"### {gene} ###")
    print(f"Kruskal-Wallis test: H={stat:.2f}, p={pval:.3e}")
    
    # ペアごとの比較 (事後解析)
    pairs = list(itertools.combinations(groups, 2))
    for g1, g2 in pairs:
        x = adata.raw[:, gene].X[adata.obs["leiden"] == g1].toarray().flatten()
        y = adata.raw[:, gene].X[adata.obs["leiden"] == g2].toarray().flatten()
        stat, pval = mannwhitneyu(x, y, alternative="two-sided")
        print(f"  {g1} vs {g2}: U={stat:.2f}, p={pval:.3e}")
    print("\n")
    
for gene in ["Vegfc"]:
    test_gene_expression(adata, gene)

### Vegfc ###
Kruskal-Wallis test: H=20.63, p=5.559e-06
  0 vs 1: U=25981.00, p=5.573e-06




In [41]:
adata.write_h5ad("./adata/Epi_Ctrl_6W_analysed.h5ad")

In [5]:
adata = sc.read_h5ad("./adata/Epi_Ctrl_6W_analysed.h5ad")

In [6]:
adata_full = adata.raw.to_adata()

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/anndata/_core/raw.py:139: FutureWarning: X.dtype being converted to np.float32 from float64. In the next version of anndata (0.9) conversion will not be automatic. Pass dtype explicitly to avoid this warning. Pass `AnnData(X, dtype=X.dtype, ...)` to get the future behavour.
  return anndata.AnnData(


In [7]:
adata_full.obs["type"]

GTCGAATCACCATATG-1    Ctrl
TCGGGCACAGCGTGCT-1    Ctrl
CGAGGCTGTAACTTCG-1    Ctrl
CTGATCCAGCTTCGTA-1    Ctrl
AGACCATCATCATCCC-1    Ctrl
                      ... 
GTCCCATCATGGGCAA-1     SED
TCGACCTAGGTCGTAG-1     SED
GGGACCTTCCATCAGA-1     SED
TCATACTCAGAGTCAG-1     SED
CCACAAATCTTCGGAA-1     SED
Name: type, Length: 416, dtype: category
Categories (3, object): ['Ctrl', 'Ex', 'SED']

# Intra sample comparison

In [45]:
adata = adata.raw.to_adata().copy()

In [46]:
adata.write_h5ad("./adata/Epi_Ctrl_6W_analysed_raw.h5ad")

In [9]:
adata = sc.read_h5ad("./adata/Epi_Ctrl_6W_analysed_raw.h5ad")

In [10]:
# Obtain cluster-specific differentially expressed gene
sc.tl.rank_genes_groups(adata, groupby="leiden",groups=("0","1"),reference="1", method="wilcoxon")

ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:00)


In [11]:
# Extract rank genes groups data from adata
rank_genes_groups = adata.uns["rank_genes_groups"]

# Convert the data into a DataFrame
groups = rank_genes_groups['names'].dtype.names  # Get all group names
dfs = []

for group in groups:
    df = pd.DataFrame({
        'gene': rank_genes_groups['names'][group],
        'logfc': rank_genes_groups['logfoldchanges'][group],
        'pvals_adj': rank_genes_groups['pvals_adj'][group]
    })
    df['group'] = group  # Add group information to identify the cell type/cluster
    dfs.append(df)

# Concatenate all groups into a single DataFrame
DEG = pd.concat(dfs, ignore_index=True)

In [12]:
DEG = DEG.loc[DEG["pvals_adj"] < 0.1]
DEG_0 = DEG.loc[DEG["logfc"] > 0.25]
DEG_1 = DEG.loc[DEG["logfc"] < -0.25]

In [16]:
DEG_0

,gene,logfc,pvals_adj,group
0,Wdr17,2.730396,1.693000e-18,0
1,Gpc3,2.450306,6.725343e-11,0
2,Tmem108,2.285828,1.073276e-10,0
3,Arhgap18,1.969315,2.807390e-09,0
4,Ptpn13,1.698067,6.119543e-09,0
...,...,...,...,...
168,Slc2a13,1.022781,9.565663e-02,0
169,Dync1i2,0.879722,9.565663e-02,0
170,Slc1a5,1.018984,9.565663e-02,0
171,Rab10,1.025661,9.611643e-02,0


In [13]:
DEG_1["logfc"] = -DEG_1["logfc"]

/var/folders/dh/5v73cjps0k72p3ltz6hhg22c0000gn/T/ipykernel_7641/2210780840.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  DEG_1["logfc"] = -DEG_1["logfc"]


In [96]:
DEG_1.to_csv("DEG_Epi_1_241024.csv")
DEG_0.to_csv("DEG_Epi_0_241024.csv")

# Enrichment analysis

In [17]:
import gseapy as gp
mouse = gp.get_library_name(organism='Mouse')
mouse

['ARCHS4_Cell-lines',
 'ARCHS4_IDG_Coexp',
 'ARCHS4_Kinases_Coexp',
 'ARCHS4_TFs_Coexp',
 'ARCHS4_Tissues',
 'Achilles_fitness_decrease',
 'Achilles_fitness_increase',
 'Aging_Perturbations_from_GEO_down',
 'Aging_Perturbations_from_GEO_up',
 'Allen_Brain_Atlas_10x_scRNA_2021',
 'Allen_Brain_Atlas_down',
 'Allen_Brain_Atlas_up',
 'Azimuth_2023',
 'Azimuth_Cell_Types_2021',
 'BioCarta_2013',
 'BioCarta_2015',
 'BioCarta_2016',
 'BioPlanet_2019',
 'BioPlex_2017',
 'CCLE_Proteomics_2020',
 'CM4AI_U2OS_Protein_Localization_Assemblies',
 'COMPARTMENTS_Curated_2025',
 'COMPARTMENTS_Experimental_2025',
 'CORUM',
 'COVID-19_Related_Gene_Sets',
 'COVID-19_Related_Gene_Sets_2021',
 'Cancer_Cell_Line_Encyclopedia',
 'Carcinogenome',
 'CellMarker_2024',
 'CellMarker_Augmented_2021',
 'ChEA_2013',
 'ChEA_2015',
 'ChEA_2016',
 'ChEA_2022',
 'Chromosome_Location',
 'Chromosome_Location_hg19',
 'ClinVar_2019',
 'ClinVar_2025',
 'DGIdb_Drug_Targets_2024',
 'DSigDB',
 'Data_Acquisition_Method_Most_Popul

In [18]:
import numpy as np

## Cluster 0 DEG

In [22]:
up_genes = DEG_0["gene"]
up_genes = up_genes.squeeze().str.strip().to_list()

In [23]:
up_genes

['Wdr17',
 'Gpc3',
 'Tmem108',
 'Arhgap18',
 'Ptpn13',
 'Efemp1',
 'Adk',
 'Il1rapl1',
 'Pdgfd',
 'Gpm6a',
 'Fmo2',
 'Sox6',
 'Rcan2',
 'Dpp4',
 'Ptprd',
 'Slc4a4',
 'Adam33',
 'Syne2',
 'Cdon',
 'Mast4',
 'Ptprq',
 'Zbtb16',
 'Pdlim5',
 'Sema3c',
 '1010001N08Rik',
 'Dock5',
 'Rapgef5',
 'Map3k5',
 'Rps21',
 'Celf2',
 'Grip1',
 'Flrt2',
 'Cdh11',
 'Cobl',
 'Dipk2a',
 'Fhit',
 'Plxna4',
 'Ctdspl',
 'Npas3',
 'Tspan5',
 'Prkg1',
 'Ccdc141',
 'Igfbp5',
 'Map4k3',
 'Slc39a8',
 'Rbbp8',
 'Mast2',
 'Aldh1a1',
 'Nr6a1',
 'Adamtsl1',
 'Kank1',
 'Dab2',
 'Colec12',
 'Crim1',
 'Man1a',
 'Sema5a',
 'Bmpr1a',
 'Ankrd12',
 'Fras1',
 'Neo1',
 'Aebp1',
 'C4b',
 'Itgbl1',
 'Slco2a1',
 'Ssh2',
 'Upk3b',
 'Clip1',
 'Zbtb20',
 'Gsta3',
 'Tead1',
 'Shroom3',
 'Col4a4',
 'Mid1',
 'Hlf',
 'Tmtc2',
 'Rarres2',
 'Rbm28',
 'Ninl',
 'Ankrd44',
 'Sulf1',
 'Rbpj',
 'Ptk2b',
 'Braf',
 'Cd200',
 'Me3',
 'Lvrn',
 'Plxdc2',
 'Kif13a',
 'Rbfox1',
 'Rras2',
 'Arid1b',
 'Efna5',
 'Klf12',
 'Lrp2',
 'Dnajc6',
 'Igf2bp2',

In [24]:
geneset_list = ['MSigDB_Hallmark_2020','GO_Biological_Process_2023','KEGG_2019_Mouse','Reactome_2022']
enr = gp.enrichr(gene_list=up_genes,
                 gene_sets=geneset_list,
                 organism='mouse', # don't forget to set organism to the one you desired! e.g. Yeast
                 outdir="./Epi_C0_GO", # don't write to disk
                )
enr.results['n_genes'] = [int(x.split('/')[0]) for x in enr.results['Overlap']]

/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(
/Users/takahiro/miniforge3/envs/scanpy/lib/python3.10/site-packages/gseapy/plot.py:694: FutureWarning: The 'method' keyword in Series.replace is deprecated and will be removed in a future version.
  df[self.colname].replace(


In [25]:
metrics_to_sort = '-log10(adjusted P-value)'
for geneset in geneset_list:
    print(geneset)
    df = enr.results[enr.results['Gene_set'] == geneset]
    df = df[df['Adjusted P-value'] < 0.05]
    df["-log10(adjusted P-value)"] = -np.log10(df['Adjusted P-value'])
    df = df.sort_values(metrics_to_sort,ascending=False)
    display(df[:30])
    
    # plot
    n_rank = 10
    plt.rcParams['axes.grid'] = False
    plt.rcParams['figure.figsize'] = 4,3
    plt.barh(width=df[:n_rank][metrics_to_sort],
             y=[x.split(' (')[0] for x in df[:n_rank]['Term']],
             color='darkred')
    plt.gca().invert_yaxis()
    plt.xlabel(metrics_to_sort)
    plt.title(f'{geneset}')
    plt.margins(y=0.02)
    plt.show()

MSigDB_Hallmark_2020


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
0,MSigDB_Hallmark_2020,UV Response Dn,8/144,0.000037,0.001478,0,0,7.019964,71.647564,DAB2;EFEMP1;IGFBP5;NFIB;CELF2;PDLIM5;CDON;BMPR1A,8,2.830461
1,MSigDB_Hallmark_2020,Mitotic Spindle,7/199,0.001763,0.035258,0,0,4.312406,27.344132,CLIP1;ARHGEF12;ARHGAP29;SSH2;RAPGEF5;PDLIM5;MID1,7,1.452747


GO_Biological_Process_2023


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
40,GO_Biological_Process_2023,Transmembrane Receptor Protein Tyrosine Kinase...,15/284,2.723890e-08,0.000035,0,0,6.902499,120.232000,RARRES2;VEGFC;EFNA5;SULF1;PRLR;EGFR;GHR;FLRT2;...,15,4.453212
41,GO_Biological_Process_2023,Regulation Of Vascular Associated Smooth Muscl...,4/12,2.535416e-06,0.001639,0,0,58.636095,755.535036,DOCK5;IGFBP5;DOCK7;PRKG1,4,2.785382
42,GO_Biological_Process_2023,Positive Regulation Of Protein Phosphorylation...,14/377,5.381628e-06,0.001896,0,0,4.721243,57.280577,RARRES2;DOCK7;VEGFC;BRAF;EFNA5;EGFR;C3;GHR;DAB...,14,2.722269
43,GO_Biological_Process_2023,Regulation Of Cell Migration (GO:0030334),15/434,5.863985e-06,0.001896,0,0,4.397450,52.974681,SEMA5A;KANK1;IGFBP5;SEMA3C;SEMA3D;SULF1;PODN;E...,15,2.722269
44,GO_Biological_Process_2023,Protein Phosphorylation (GO:0006468),16/500,7.502323e-06,0.001940,0,0,4.072854,48.060886,CAMK1D;MAST4;MAST2;STK39;BRAF;EGFR;LATS2;EFEMP...,16,2.712176
45,GO_Biological_Process_2023,Positive Regulation Of Apoptotic Cell Clearanc...,3/7,2.170124e-05,0.004009,0,0,87.454412,939.097811,C3;C4B;C2,3,2.397015
46,GO_Biological_Process_2023,Positive Regulation Of Vascular Associated Smo...,3/7,2.170124e-05,0.004009,0,0,87.454412,939.097811,DOCK5;IGFBP5;DOCK7,3,2.397015
47,GO_Biological_Process_2023,Regulation Of Actin Filament Polymerization (G...,6/70,3.184122e-05,0.005146,0,0,11.094499,114.880747,KANK1;BAIAP2L1;ARHGAP18;PTK2B;PAK3;SSH2,6,2.288502
48,GO_Biological_Process_2023,Axonogenesis (GO:0007409),9/188,3.899153e-05,0.005602,0,0,6.023709,61.153694,SEMA5A;RAB10;SEMA3C;SEMA3D;DOCK7;EFNA5;PAK3;EP...,9,2.251674
49,GO_Biological_Process_2023,Axon Guidance (GO:0007411),8/149,4.711300e-05,0.006092,0,0,6.769310,67.442377,SEMA5A;SEMA3C;SEMA3D;NFIB;EFNA5;EPHB1;NEO1;PLXNA4,8,2.215261


KEGG_2019_Mouse


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
1333,KEGG_2019_Mouse,Axon guidance,10/180,0.000004,0.000634,0,0,7.093829,88.366301,SEMA5A;ARHGEF12;SEMA3C;SEMA3D;EFNA5;PAK3;SSH2;...,10,3.197725
1334,KEGG_2019_Mouse,Protein digestion and absorption,6/90,0.000131,0.010687,0,0,8.444397,75.487585,DPP4;ELN;COL4A4;COL4A5;SLC1A5;ATP1B1,6,1.971161
1335,KEGG_2019_Mouse,Focal adhesion,8/199,0.000348,0.018895,0,0,4.984547,39.696890,PDGFD;COL4A4;KDR;VEGFC;COL4A5;BRAF;PAK3;EGFR,8,1.723651
1336,KEGG_2019_Mouse,Ras signaling pathway,8/233,0.000980,0.030593,0,0,4.224000,29.264755,PDGFD;KDR;VEGFC;RRAS2;EFNA5;PAK3;RAPGEF5;EGFR,8,1.514380
1337,KEGG_2019_Mouse,MAPK signaling pathway,9/294,0.001076,0.030593,0,0,3.762901,25.718903,PDGFD;KDR;VEGFC;RRAS2;BRAF;EFNA5;EGFR;MAP4K3;M...,9,1.514380
1338,KEGG_2019_Mouse,PI3K-Akt signaling pathway,10/357,0.001126,0.030593,0,0,3.444069,23.381715,GHR;PDGFD;COL4A4;KDR;VEGFC;COL4A5;IL6RA;EFNA5;...,10,1.514380
1339,KEGG_2019_Mouse,Proteoglycans in cancer,7/203,0.001973,0.045951,0,0,4.223537,26.304317,ARHGEF12;KDR;GPC3;ITPR1;RRAS2;BRAF;EGFR,7,1.337710
1340,KEGG_2019_Mouse,Rap1 signaling pathway,7/209,0.002325,0.047371,0,0,4.096833,24.843442,PDGFD;KDR;VEGFC;BRAF;EFNA5;RAPGEF5;EGFR,7,1.324490


Reactome_2022


,Gene_set,Term,Overlap,P-value,Adjusted P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,n_genes,-log10(adjusted P-value)
1496,Reactome_2022,RHO GTPase Cycle R-HSA-9012999,14/441,0.000031,0.016462,0,0,4.000412,41.522604,DOCK5;ARHGEF12;KIDINS220;DOCK7;ARHGAP29;ARHGAP...,14,1.783519
1497,Reactome_2022,Signaling By Receptor Tyrosine Kinases R-HSA-9...,14/496,0.000109,0.027086,0,0,3.533887,32.234533,KIDINS220;DOCK7;ITPR1;VEGFC;BRAF;EGFR;FLRT2;PD...,14,1.567256
1498,Reactome_2022,Signaling By Rho GTPases R-HSA-194315,16/644,0.000155,0.027086,0,0,3.115583,27.332733,DOCK5;DYNC1I2;ARHGEF12;KIDINS220;DOCK7;ARHGAP2...,16,1.567256
1499,Reactome_2022,"Signaling By Rho GTPases, Miro GTPases And RHO...",16/660,0.000204,0.027086,0,0,3.035645,25.788784,DOCK5;DYNC1I2;ARHGEF12;KIDINS220;DOCK7;ARHGAP2...,16,1.567256
1500,Reactome_2022,Signal Transduction R-HSA-162582,38/2465,0.000256,0.027102,0,0,2.018038,16.692376,KANK1;DOCK5;DYNC1I2;DOCK7;ITPR1;ARHGAP18;SLC1A...,38,1.566999
